# Train Int8 CNN And Export PE3x4 Cluster Testcases

Goal of this notebook:

1. Build and train a small multi-layer CNN in PyTorch.
2. Quantize activations and weights into integer tensors.
3. Run the reference convolution in software using dense PyTorch `conv2d`.
4. Export `.txt` testcase files that the SystemVerilog testbench can read later through `+CNN_CASE_FILE=<path>`.
5. Preview the generated dense tensors, CSC representation, golden PSUM, and int8 PPU golden.

Important separation:

- This notebook/script creates testcase files.
- The SystemVerilog testbench reads those files later and runs the hardware cluster.
- The Python golden is dense convolution, not a copy of the sparse hardware dataflow.

## 1. Setup

This cell finds the repository root, imports PyTorch, and imports the testcase generator as a normal Python module. The generator contains the shared helper functions used by both this notebook and the standalone export script.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

import torch
import torch.nn.functional as F

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SCRIPT = ROOT / "scripts" / "generate_int8_cnn_cluster_cases.py"
OUT_DIR = ROOT / "tb" / "generated_cnn_cases"
REGRESSION_SCRIPT = ROOT / "scripts" / "run_cluster_system_cnn_case_regression.ps1"

spec = importlib.util.spec_from_file_location("casegen", SCRIPT)
casegen = importlib.util.module_from_spec(spec)
sys.modules["casegen"] = casegen
spec.loader.exec_module(casegen)

print("ROOT             =", ROOT)
print("generator script =", SCRIPT)
print("output folder    =", OUT_DIR)
print("torch version    =", torch.__version__)

## 2. CNN model used to generate realistic tensors

The model is intentionally small, but it has multiple convolution layers. Testcases are extracted from different source layers so the exported IACT tensors are not all identical. The hardware testcase still targets one 3x3 convolution at a time, because the current PE3x4 cluster verification target is a single stride-1 row-stationary convolution layer.

In [ ]:
model = casegen.TinyMultiLayerCNN()
print(model)

print("\nParameter shapes:")
for name, param in model.named_parameters():
    print(f"  {name:16s} shape={tuple(param.shape)}")

## 3. Training data and short training run

The dataset is synthetic and deterministic. It is only used to produce non-trivial activation and weight tensors. The goal is not model accuracy; the goal is to generate realistic, varied convolution tensors for hardware verification.

In [ ]:
torch.manual_seed(7)
model = casegen.TinyMultiLayerCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
train_x, train_y = casegen.make_dataset(n=128)

print("train_x shape =", tuple(train_x.shape))
print("train_y shape =", tuple(train_y.shape))

for epoch in range(12):
    optimizer.zero_grad()
    logits = model(train_x)
    loss = F.cross_entropy(logits, train_y)
    loss.backward()
    optimizer.step()
    if epoch in [0, 1, 2, 5, 11]:
        pred = logits.argmax(dim=1)
        acc = (pred == train_y).float().mean().item()
        print(f"epoch={epoch:02d} loss={loss.item():.5f} acc={acc:.3f}")

model.eval()

## 4. Quantization rule

Weights use symmetric integer quantization. Activations use a simple integer activation quantization. These integer tensors are what the testcase exports. The dense software reference then runs `torch.nn.functional.conv2d` on these quantized integer tensors.

In [ ]:
source_layer = 0
sample_index = 0
cfg = casegen.case_configs()[0]

iact_full = casegen.conv_layer_activation(model, train_x, source_layer, sample_index)
weight_full = casegen.conv_layer_weights(model, source_layer)

iact_q = iact_full[:cfg.c_in, :cfg.h, :cfg.w].clone()
weight_q = weight_full[:cfg.m_out, :cfg.c_in, :, :].clone()

print("Example case config:", cfg)
print("Quantized IACT shape :", tuple(iact_q.shape), "min/max=", int(iact_q.min()), int(iact_q.max()))
print("Quantized Weight shape:", tuple(weight_q.shape), "min/max=", int(weight_q.min()), int(weight_q.max()))
print("\nIACT C0 preview:")
print(iact_q[0])
print("\nWeight M0 preview:")
print(weight_q[0])

## 5. Dense PyTorch convolution golden

This is the key correctness reference. The golden PSUM is computed with dense software convolution. It does not use the hardware sparse scheduler, CSC traversal, PE mapping, or row-stationary mask logic.

In [ ]:
conv_golden = F.conv2d(iact_q.unsqueeze(0).float(), weight_q.float()).round().to(torch.int32)[0]
print("Golden PSUM tensor shape =", tuple(conv_golden.shape))
print("Golden PSUM M0:")
print(conv_golden[0])

## 6. PPU int8 golden

The hardware cluster produces PSUM. Because this flow uses int8-style tensors, the testbench also sends the hardware PSUM into the existing PPU and compares against this Python int8 golden. The PPU step is separate from convolution: convolution correctness is checked at PSUM level first, then requantized int8 output is checked.

In [ ]:
psum_seed = casegen.psum_seed_tensor(cfg.m_out, cfg.h - 2, cfg.w - 2, cfg.psum_seed, case_id=0)
psum_golden = conv_golden + psum_seed
ppu_bias = torch.tensor([((0 + 1) * (m + 1)) % 17 - 8 for m in range(cfg.m_out)], dtype=torch.int32)
ppu_m0 = 1 << 30
ppu_n = 0
ppu_z_out = 0
int8_golden = casegen.ppu_requantize_tensor(psum_golden, ppu_bias, ppu_m0, ppu_n, ppu_z_out)

print("PPU config: bias=", ppu_bias.tolist(), "M0=", ppu_m0, "n=", ppu_n, "z_out=", ppu_z_out)
print("PSUM golden M0:")
print(psum_golden[0])
print("\nInt8 golden M0:")
print(int8_golden[0])

## 7. CSC preview

The exported JSON includes CSC-style views for inspection. This is useful for checking sparsity and testcase contents. The golden is still dense PyTorch convolution; CSC is only a representation of the same quantized tensors.

In [ ]:
iact_csc = casegen.csc_iact_by_column(iact_q)
weight_csc = casegen.csc_weight_generic(weight_q)

print("First 4 IACT CSC streams:")
for stream in iact_csc[:4]:
    print(stream)

print("\nFirst 4 Weight CSC streams:")
for stream in weight_csc[:4]:
    print(stream)

## 8. Export 20 testcase `.txt` files

This is the main output of the notebook. The generator writes one `.txt` file per testcase. Each file contains only simple records such as `CONFIG`, `IACT`, `WEIGHT`, `PSUM`, `GOLDEN`, and `GOLDEN_INT8`. The SystemVerilog testbench reads this text file later; it does not need to know PyTorch.

In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT), "--out-dir", str(OUT_DIR)],
    cwd=str(ROOT),
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

manifest = json.loads((OUT_DIR / "manifest.json").read_text())
print("Generated case_count =", manifest["case_count"])
print("Case files are in   =", OUT_DIR)

## 9. Inspect generated cases

This cell prints a compact manifest and previews one generated `.txt` testcase. This is the exact text format consumed by the testbench.

In [ ]:
manifest = json.loads((OUT_DIR / "manifest.json").read_text())

print("Manifest summary:")
for item in manifest["cases"][:8]:
    print(item)
print("... total cases:", manifest["case_count"])

case_id = 0
case_txt = OUT_DIR / f"cnn_case_{case_id:02d}.txt"
print(f"\nText testcase preview: {case_txt}")
print("-" * 80)
print("\n".join(case_txt.read_text().splitlines()[:40]))

## 10. Inspect dense tensors and expected outputs from one exported case

The `.json` sidecar is for human inspection only. The testbench reads the `.txt` file. Here we load one JSON file to show IACT, Weight, PSUM golden, and PPU int8 golden in a readable way.

In [ ]:
case_id = 1
case_json = json.loads((OUT_DIR / f"cnn_case_{case_id:02d}.json").read_text())

print("config:", case_json["config"])
print("oracle:", case_json["oracle"])
print("\nIACT C0:")
for row in case_json["iact_dense"][0]:
    print(row)
print("\nWeight M0:")
for c, mat in enumerate(case_json["weight_dense"][0]):
    print("C", c)
    for row in mat:
        print(row)
print("\nGolden PSUM M0:")
for row in case_json["golden"][0]:
    print(row)
print("\nGolden int8 after PPU M0:")
for row in case_json["ppu"]["golden_int8"][0]:
    print(row)

## 11. Run hardware regression after testcase export

Run this after the `.txt` files are generated. The script compiles the cluster-system testbench and runs all generated cases. The log prints hardware PSUM samples, Python golden samples, PPU int8 samples, and final PASS/FAIL status.

You can run it from PowerShell outside the notebook:

```powershell
cd "D:\Eyeriss v2 Accelerator\PE\hdl"
powershell -ExecutionPolicy Bypass -File scripts\run_cluster_system_cnn_case_regression.ps1
```

Or run it from this cell.

In [ ]:
print("Regression command:")
print(f"powershell -ExecutionPolicy Bypass -File {REGRESSION_SCRIPT}")

# Uncomment this line when you want to run the full SystemVerilog simulation from the notebook.
# subprocess.run(["powershell", "-ExecutionPolicy", "Bypass", "-File", str(REGRESSION_SCRIPT)], cwd=str(ROOT), check=True)